# Домашнее задание 5. Ньютон–Лагранж и обобщённый Гаусс–Ньютон

**Выдано:** 11 ноября 2026. **Срок сдачи:** 25 ноября 2026 (до начала занятия). **Максимум:** 10 баллов (+2 бонус).

**Главная идея.** Задачу с ограничениями-равенствами решают как систему уравнений: условие касания $\nabla f=J^\top\lambda$ и сами ограничения $g=0$ (лекция 8). Метод Ньютона–Лагранжа — метод Ньютона для этой системы. На каждом шаге он решает одну линейную систему с матрицей ККТ (лекция 9). Обобщённый Гаусс–Ньютон ставит в эту матрицу $J_r^\top J_r$ вместо гессиана лагранжиана. Вы напишете оба метода и сравните их с IPOPT на одной задаче — руке робота. Вы увидите, когда каждый метод сходится быстро, а когда ломается.

**Как сдавать.** Работа сдаётся в ваш личный закрытый репозиторий `optimisation-misis-2026/numopt-<ваш логин>`; этот ноутбук уже лежит там как `hw5/homework.ipynb`. Код пишите в ячейках с `# TODO`, ответы — в markdown-ячейках вместо `_Ваш ответ:_ TODO`: 2–4 предложения с числами из ваших выводов. Выполните ноутбук целиком (`Kernel → Restart Kernel and Run All Cells`). Загрузите его в папку `hw5/` под тем же именем — пошагово в [инструкции](https://github.com/IlyaChichkanov/Numerical-Optimization/blob/main/CONTRIBUTING.md). Задачи можно обсуждать, но текст и код пишутся самостоятельно; своё решение никому не показывайте.

**Материал:** лекция 8 ([конспект](https://github.com/IlyaChichkanov/Numerical-Optimization/blob/main/lectures/lecture08/lecture08.md)) — условия Лагранжа, LICQ, редуцированный гессиан; лекция 9 ([конспект](https://github.com/IlyaChichkanov/Numerical-Optimization/blob/main/lectures/lecture09/lecture09.md), [демо](https://github.com/IlyaChichkanov/Numerical-Optimization/blob/main/lectures/lecture09/demo09.ipynb)) — оба метода и матрица ККТ. Ничего сверх лекций не требуется. Ориентировочное время — около двух часов, бонус — ещё около получаса.

**Устройство задания.** Задача, её производные (их строит CasADi), IPOPT и рисовальщики даны готовыми. Вы пишете две короткие функции: `newton_lagrange` в задаче 1 и `gn_kkt` в задаче 2. Под каждой стоит проверка `assert` на числах конспекта лекции 9: прошла — можно идти дальше. Остальное — запуски, картинки и ответы на вопросы по ним.

Запуск: в Google Colab первая ячейка сама поставит CasADi; локально — `uv sync --group casadi` в корне клона репозитория курса, затем `uv run jupyter lab`.

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("casadi") is None:          # Colab: поставить CasADi один раз
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "casadi"], check=True)

import casadi as ca
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import null_space

BLUE, ORANGE, AQUA, YELLOW, RED, VIOLET, GRAY, INK = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e34948", "#4a3aa7", "#8a8985", "#0b0b0b"
COLORS = [BLUE, ORANGE, AQUA, VIOLET]      # цвета кривых по порядку: первая — синяя, вторая — оранжевая, ...
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.2, "legend.frameon": False})


# ---------------------------------------------------------------- рука робота: три звена длины 1, углы th = (th1, th2, th3)
_th = ca.SX.sym("th", 3); _lam = ca.SX.sym("lam", 2)
_t = ca.SX.sym("t", 2); _c = ca.SX.sym("c", 3)                    # цель t и текущая поза th_c — параметры задачи
_phi = ca.cumsum(_th)                                              # углы звеньев с осью x: th1, th1 + th2, th1 + th2 + th3
_e = ca.vertcat(ca.sum1(ca.cos(_phi)), ca.sum1(ca.sin(_phi)))      # конец руки e(th)
_f = 0.5 * ca.sumsqr(_th - _c)                                     # f = 1/2 |th - th_c|^2
_g = _e - _t                                                       # g = e(th) - t: два уравнения, по x и по y
_kkt_fun = ca.Function("arm_kkt", [_th, _lam, _t, _c],             # производные строит CasADi (лекция 7)
                       [ca.gradient(_f, _th), ca.jacobian(_g, _th), ca.hessian(_f - ca.dot(_lam, _g), _th)[0], _g])
_end_fun = ca.Function("arm_end", [_th], [_e])
_ipopt = ca.nlpsol("ipopt", "ipopt", {"x": _th, "p": ca.vertcat(_t, _c), "f": _f, "g": _g},
                   {"ipopt.print_level": 0, "print_time": 0, "ipopt.sb": "yes", "ipopt.tol": 1e-10})

TH_C = np.array([0.3, 0.9, 0.9])          # текущая поза, рад
T1 = np.array([1.0, 1.5])                 # цель задачи 1(а)


def arm_end(th):
    """Конец руки e(th) — точка плоскости."""
    return np.array(_end_fun(th)).ravel()


def arm_joints(th):
    """Точки руки по порядку: основание, второй и третий суставы, конец — массив 4 x 2."""
    phi = np.cumsum(th)
    return np.vstack([[0.0, 0.0], np.cumsum(np.c_[np.cos(phi), np.sin(phi)], axis=0)])


def arm_kkt(t, theta_c=TH_C):
    """Производные задачи min 1/2 |th - theta_c|^2 при e(th) - t = 0 в знаке L = f - lam^T g.
    Возвращает функцию kkt(th, lam) -> (grad f, J, H, g): grad f и g — векторы, J — 2 x 3, H = гессиан L по th — 3 x 3."""
    t, theta_c = np.asarray(t, float), np.asarray(theta_c, float)

    def kkt(th, lam):
        gf, J, H, g = _kkt_fun(th, lam, t, theta_c)
        return np.array(gf).ravel(), np.array(J), np.array(H), np.array(g).ravel()
    return kkt


def solve_ipopt(t, theta_c=TH_C, x0=None):
    """IPOPT для той же задачи из x0 (по умолчанию — из theta_c). Возвращает (th*, lam* в нашем знаке, итераций)."""
    theta_c = np.asarray(theta_c, float)
    sol = _ipopt(x0=theta_c if x0 is None else x0, p=np.concatenate([t, theta_c]), lbg=0, ubg=0)
    return np.array(sol["x"]).ravel(), -np.array(sol["lam_g"]).ravel(), _ipopt.stats()["iter_count"]   # IPOPT: L = f + lam_g^T g


def target_rotated(beta):
    """Нынешний конец руки e(TH_C), повёрнутый вокруг основания на угол beta (рад)."""
    c, s = np.cos(beta), np.sin(beta); e = arm_end(TH_C)
    return np.array([c * e[0] - s * e[1], s * e[0] + c * e[1]])


# ---------------------------------------------------------------- окружность лекции 9 — для проверок
def circle_kkt(a):
    """Ближайшая к a точка единичной окружности: f = 1/2 |x - a|^2, g = 1/2 (|x|^2 - 1); grad f = x - a, J = x^T, H = (1 - lam) I."""
    def kkt(x, lam):
        return x - a, x[None, :], (1 - lam[0]) * np.eye(2), np.array([0.5 * (x @ x - 1)])
    return kkt


# ---------------------------------------------------------------- рисовальщики
def draw_arm(ax, th, color=BLUE, alpha=1.0, lw=3.0, label=None):
    """Рука в позе th: звенья линией, суставы и конец кружками."""
    P = arm_joints(th)
    ax.plot(P[:, 0], P[:, 1], "-", color=color, lw=lw, alpha=alpha, solid_capstyle="round", label=label, zorder=4)
    ax.plot(P[1:, 0], P[1:, 1], "o", color=color, ms=6, alpha=alpha, mec="white", zorder=5)


def _arm_axes(ax, t, title, reach=False):
    if reach:
        s = np.linspace(0, 2 * np.pi, 400)
        ax.plot(3 * np.cos(s), 3 * np.sin(s), "--", color=GRAY, lw=1, label="граница досягаемости")
    ax.plot(0, 0, "s", color=INK, ms=10, zorder=6)
    ax.plot(*t, "*", color=RED, ms=17, mec="white", zorder=7, label="цель $t$")
    ax.set(aspect="equal", xlabel="$x$", ylabel="$y$", title=title)


def plot_problem(theta_c, t):
    """Картинка задачи: рука в позе theta_c и цель t."""
    _, ax = plt.subplots(figsize=(5.5, 5))
    draw_arm(ax, theta_c, color=BLUE, label=r"текущая поза $\theta_c$")
    _arm_axes(ax, t, "Рука и цель")
    ax.legend(fontsize=8, loc="upper left")
    return ax


def plot_nl(hist, t):
    """Картинка 1. Слева — позы руки по итерациям (старт серый, дальше всё темнее) и путь конца руки; справа — |F_k|."""
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12.5, 4.8))
    n = len(hist)
    for k, (x, _, _) in enumerate(hist):
        draw_arm(a1, x, color=GRAY if k == 0 else BLUE, alpha=1.0 if k == 0 else 0.3 + 0.7 * k / (n - 1), lw=2.5)
    ends = np.array([arm_end(x) for x, _, _ in hist])
    a1.plot(ends[:, 0], ends[:, 1], ":", color=ORANGE, lw=1.6, zorder=6)
    if n > 1:                                                          # старт подписан в легенде, здесь — первая итерация
        a1.annotate("$k=1$", ends[1], textcoords="offset points", xytext=(-38, -2), fontsize=9, color=INK)
    a1.plot([], [], "-", color=GRAY, lw=2.5, label="старт, $k=0$"); a1.plot([], [], "-", color=BLUE, lw=2.5, label="итерации")
    a1.plot([], [], ":", color=ORANGE, lw=1.6, label="путь конца руки")
    _arm_axes(a1, t, "Позы по итерациям")
    a1.legend(fontsize=8, loc="upper left")
    a2.semilogy(np.arange(n), [h[2] for h in hist], "-o", color=BLUE, ms=5)
    a2.set(xlabel="итерация $k$", ylabel=r"$\|F_k\|$", title="Невязка по итерациям")
    plt.tight_layout()
    return a1, a2


def plot_reach(deltas, lam_norms, conds, theta, t):
    """Картинка 2. Слева — поза theta для цели t у границы; справа — |lam*| и обусловленность K от delta = 3 - d."""
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12.5, 4.8))
    draw_arm(a1, theta, color=BLUE, label="ответ")
    _arm_axes(a1, t, "Цель у границы", reach=True)
    a1.legend(fontsize=8, loc="lower left")
    a2.loglog(deltas, lam_norms, "-o", color=BLUE, label=r"$\|\lambda^*\|$")
    a2.loglog(deltas, conds, "-s", color=ORANGE, label="обусловленность $K$")
    a2.invert_xaxis()
    a2.set(xlabel=r"расстояние до границы $\delta=3-d$", ylabel="значение", title="Множители и обусловленность")
    a2.legend(fontsize=9)
    plt.tight_layout()
    return a1, a2


def plot_errors(err_gn, err_nl, labels):
    """Картинка 3. Ошибка |th_k - th*| по итерациям: Гаусс–Ньютон — сплошные линии, Ньютон–Лагранж — пунктир того же цвета."""
    _, ax = plt.subplots(figsize=(11, 4.6))
    for i, (eg, en, lab) in enumerate(zip(err_gn, err_nl, labels)):
        eg, en = np.asarray(eg), np.asarray(en)
        kg, kn = np.nonzero(eg > 0)[0], np.nonzero(en > 0)[0]          # нулевую ошибку на лог. шкале не показать
        ax.semilogy(kg, eg[kg], "-o", color=COLORS[i], ms=4, lw=1.6, label=f"{lab}: Гаусс–Ньютон")
        ax.semilogy(kn, en[kn], "--s", color=COLORS[i], ms=4, lw=1.2, alpha=0.75, label=f"{lab}: Ньютон–Лагранж")
    ax.set(xlabel="итерация $k$", ylabel=r"ошибка $\|\theta_k-\theta^*\|$", title="Ошибка по итерациям")
    ax.legend(fontsize=8, loc="upper left", bbox_to_anchor=(1.01, 1.0))
    plt.tight_layout()
    return ax


def plot_track(poses, targets, its_gn, its_nl):
    """Бонус, картинка 4. Слева — позы руки по кадрам (всё темнее) и цели кадров; справа — итерации на кадр у двух методов."""
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12.5, 4.8), gridspec_kw={"width_ratios": [1.1, 1]})
    n = len(poses)
    for k, th in enumerate(poses):
        draw_arm(a1, th, color=GRAY if k == 0 else BLUE, alpha=1.0 if k == 0 else 0.2 + 0.8 * k / (n - 1), lw=2)
    T = np.array(targets)
    a1.plot(T[:, 0], T[:, 1], ".", color=RED, ms=7, zorder=6, label="цели кадров")
    _arm_axes(a1, T[-1], "Рука по кадрам")
    a1.legend(fontsize=8, loc="lower left")
    frames = np.arange(1, len(its_gn) + 1)
    a2.bar(frames - 0.2, its_gn, width=0.4, color=BLUE, label="Гаусс–Ньютон")
    a2.bar(frames + 0.2, its_nl, width=0.4, color=ORANGE, label="Ньютон–Лагранж")
    a2.set(xlabel="кадр", ylabel="итераций", title="Итерации на кадр", ylim=(0, 1.3 * max(max(its_gn), max(its_nl))))
    a2.legend(fontsize=8, loc="upper center", ncol=2)
    plt.tight_layout()
    return a1, a2


print(f"конец руки в текущей позе: e(TH_C) = {arm_end(TH_C).round(3)};  цель задачи 1(а): T1 = {T1}")

## Задача задания: рука робота

**Рука.** Плоский манипулятор из трёх звеньев длины 1 закреплён в точке $(0,0)$. Его поза — вектор углов $\theta=(\theta_1,\theta_2,\theta_3)\in\mathbb R^3$ в радианах. Здесь $\theta_1$ — угол первого звена с осью $x$, а $\theta_2$ и $\theta_3$ — углы во втором и третьем суставах. Звено $k$ повёрнуто к оси $x$ на угол $\varphi_k=\theta_1+\dots+\theta_k$. Поэтому конец руки — в точке

$$
e(\theta)=\begin{pmatrix}\cos\varphi_1+\cos\varphi_2+\cos\varphi_3\\ \sin\varphi_1+\sin\varphi_2+\sin\varphi_3\end{pmatrix}.
$$

**Задача.** Рука стоит в позе $\theta_c=(0.3,\ 0.9,\ 0.9)$. Нужно привести её конец в цель $t\in\mathbb R^2$ и повернуть суставы как можно меньше:

$$
\min_{\theta\in\mathbb R^3}\ f(\theta)=\tfrac12\lVert\theta-\theta_c\rVert^2\qquad\text{при}\qquad g(\theta)=e(\theta)-t=0 .
$$

Ограничений два — по координатам $x$ и $y$:

$$
\cos\varphi_1+\cos\varphi_2+\cos\varphi_3=t_1,\qquad \sin\varphi_1+\sin\varphi_2+\sin\varphi_3=t_2 .
$$

*Размеры:* $n=3$ неизвестных, $m=2$ ограничения, якобиан $J$ — матрица $2\times3$, множители $\lambda\in\mathbb R^2$. Касательное пространство одномерно ($n-m=1$). Поэтому $Z$ — один единичный вектор, а редуцированный гессиан $Z^\top HZ$ — число.

Это обратная задача кинематики (*inverse kinematics*): по положению конца найти углы. У руки из трёх звеньев таких поз бесконечно много, и $f$ выбирает ближайшую к текущей.

**Знаки — как в лекции 9.** Лагранжиан $L=f-\lambda^\top g$, гессиан $H=\nabla^2_{\theta\theta}L$ — матрица $3\times3$. Матрица ККТ и невязка системы:

$$
K=\begin{pmatrix}H&-J^\top\\ J&0\end{pmatrix},\qquad F=\begin{pmatrix}\nabla f-J^\top\lambda\\ g\end{pmatrix}.
$$

**Что дано.** Функция `arm_kkt(t)` возвращает функцию `kkt(theta, lam)`, а она — четвёрку $(\nabla f,\ J,\ H,\ g)$ в numpy. Функция `solve_ipopt(t)` решает ту же задачу IPOPT из $\theta_c$ и возвращает $(\theta^\ast,\ \lambda^\ast,\ \text{число итераций})$. Множители там уже в нашем знаке: `lam_g` у IPOPT равен $-\lambda$.

In [ ]:
plot_problem(TH_C, T1); plt.show()          # картинка задачи: рука в позе TH_C и цель T1

## Задача 1. Метод Ньютона–Лагранжа (5 баллов)

**Главная идея.** Метод Ньютона–Лагранжа ищет $\theta$ и $\lambda$ вместе. На шаге он решает систему $K\,(p,\Delta\lambda)=-F$ и сдвигает обе части (конспект §2.2). Вблизи решения он сходится квадратично, если матрица $K$ невырождена. По теореме §3 для этого достаточно LICQ (строки $J$ линейно независимы) и $Z^\top HZ\succ0$.

**Задание 1 (1.5 балла): `newton_lagrange(kkt, x0, lam0, tol=1e-10, maxit=50)`** — алгоритм конспекта §2.2. Здесь $x\in\mathbb R^n$ — точка, $\lambda\in\mathbb R^m$ — множители. Для $k=0,1,2,\dots$:

1. посчитать `gf, J, H, g = kkt(x, lam)` и невязку $F=(\nabla f-J^\top\lambda,\ g)$ — вектор из $n+m$ чисел;
2. если $\lVert F\rVert\le$ `tol` или $k=$ `maxit` — стоп;
3. собрать $K$ (функция `np.block`) и решить $K\,(p,\Delta\lambda)=-F$;
4. сделать шаг: $x\leftarrow x+p$, $\lambda\leftarrow\lambda+\Delta\lambda$.

Верните список троек $(x_k,\ \lambda_k,\ \lVert F_k\rVert)$ для $k=0,1,\dots$, начиная со старта: его читают рисовальщики. Число итераций — длина списка минус один.

In [ ]:
def newton_lagrange(kkt, x0, lam0, tol=1e-10, maxit=50):
    """Метод Ньютона–Лагранжа, знак L = f - lam^T g. Возвращает список (x_k, lam_k, |F_k|), начиная со старта."""
    # TODO: ваш код
    raise NotImplementedError

In [ ]:
# проверка задания 1: окружность лекции 9 §2.1 — ближайшая к a = (2, 1) точка, старт (1, 1), lambda = 0
hist = newton_lagrange(circle_kkt(np.array([2.0, 1.0])), [1.0, 1.0], [0.0])
assert np.allclose(hist[1][0], [1.25, 0.25]) and np.isclose(hist[1][1][0], -0.75), \
    f"в конспекте x1 = (1.25, 0.25), lambda1 = -0.75, а у вас {hist[1][0]}, {hist[1][1]}: проверьте блок -J^T и правую часть -F"
assert len(hist) - 1 == 6 and np.allclose([h[2] for h in hist[:6]], [1.118, 0.6702, 0.1448, 0.01828, 1.643e-4, 9.47e-9], rtol=1e-3), \
    f"в конспекте 6 итераций и |F| = 1.12, 0.67, 0.14, 0.018, 1.6e-4, 9.5e-9; у вас {len(hist) - 1}: {[f'{h[2]:.2g}' for h in hist]}"
print("задание 1: ok;  |F_k| =", [f"{h[2]:.2g}" for h in hist])

**(а) Картинка 1 (1.5 балла): рука идёт к цели.** Цель $t=(1,\ 1.5)$ лежит в переменной `T1`. Запустите метод из текущей позы: `newton_lagrange(arm_kkt(T1), TH_C, np.zeros(2))`. Нулевые стартовые множители здесь совпадают с МНК: в $\theta_c$ градиент $\nabla f=\theta_c-\theta_c=0$. Нарисуйте `plot_nl(hist, T1)`: слева позы руки по итерациям, справа $\lVert F_k\rVert$. В ответе найдите $Z$ (`null_space(J)`) и число $Z^\top HZ$. Решите ту же задачу `solve_ipopt(T1)`.

Ответьте: сколько итераций и как падает $\lVert F_k\rVert$ — какая это сходимость? Попадает ли конец руки в цель на промежуточных итерациях и почему? Минимум ли ответ? Совпадают ли с IPOPT ответ, множители и число итераций?

*Полный балл:* картинка; итерации и $\lVert F_k\rVert$; положение конца руки по итерациям; $Z^\top HZ$ с выводом о минимуме; сверка с IPOPT. *Половина:* картинка и числа без объяснений.

In [ ]:
# TODO: ваш код
# hist1 = newton_lagrange(arm_kkt(T1), TH_C, np.zeros(2)); напечатайте число итераций, |F_k| и arm_end(theta_k) по итерациям
# в ответе: gf, J, H, g = arm_kkt(T1)(theta, lam); Z = null_space(J); число (Z.T @ H @ Z).item()
# IPOPT: th_ip, lam_ip, it_ip = solve_ipopt(T1)
# картинка: plot_nl(hist1, T1); plt.show()


_Ваш ответ:_ TODO

**(б) Картинка 2 (2 балла): цель у границы досягаемости.** Рука из трёх звеньев длины 1 достаёт не дальше чем на $3$ от основания. Поставьте цель на луче под углом $0.6$ рад: $t=d\,(\cos0.6,\ \sin0.6)$, где $d=3-\delta$ и $\delta=0.5,\ 0.1,\ 10^{-2},\ 10^{-3},\ 10^{-4}$. Для каждой цели запустите метод из $\theta_c$ с нулевыми множителями. Запишите число итераций, $\lVert\lambda^\ast\rVert$ и число обусловленности $K$ в ответе (`np.linalg.cond`, лекция 4). Нарисуйте `plot_reach(deltas, lam_norms, conds, theta, t)` — поза для последней цели и обе кривые. Затем запустите наш метод и IPOPT для $d=3$. Если `np.linalg.solve` сообщит о вырожденной матрице, это тоже ответ.

Ответьте: во сколько раз растут $\lVert\lambda^\ast\rVert$ и обусловленность $K$, когда $\delta$ уменьшается в $10$ раз? Выпишите на бумаге якобиан $J$ в вытянутой позе $\theta=(0.6,\ 0,\ 0)$ и найдите его ранг. Какое условие теоремы §3 нарушено при $d=3$? Почему множители растут уже на подходе к границе?

*Полный балл:* картинка и таблица чисел; $J$ в вытянутой позе и его ранг; объяснение через LICQ и разложение $\nabla f=J^\top\lambda$ по почти параллельным строкам $J$; что сделали оба метода при $d=3$. *Половина:* числа и картинка без объяснения.

In [ ]:
# TODO: ваш код
# U = (cos 0.6, sin 0.6); для delta из (0.5, 0.1, 1e-2, 1e-3, 1e-4): t = (3 - delta) U, h = newton_lagrange(arm_kkt(t), TH_C, np.zeros(2))
# в ответе: итерации, |lambda*|, K = np.block(...) и np.linalg.cond(K); соберите списки lam_norms и conds
# J в вытянутой позе: arm_kkt(3 * U)(np.array([0.6, 0, 0]), np.zeros(2))[1]
# d = 3: newton_lagrange(arm_kkt(3 * U), TH_C, np.zeros(2)) и solve_ipopt(3 * U)
# картинка: plot_reach(deltas, lam_norms, conds, theta, t); plt.show() — theta и t для delta = 1e-4


_Ваш ответ:_ TODO

## Задача 2. Обобщённый Гаусс–Ньютон (5 баллов)

**Главная идея.** Цель $f=\tfrac12\lVert r\rVert^2$ — сумма квадратов невязок $r(\theta)=\theta-\theta_c$, а якобиан невязок $J_r=I$. Обобщённый Гаусс–Ньютон ставит в матрицу ККТ $J_r^\top J_r=I$ вместо точного

$$
H=I-\lambda_1\nabla^2e_1-\lambda_2\nabla^2e_2 .
$$

Он выбрасывает кривизну ограничений (конспект §5), поэтому вторые производные ограничений и множители для матрицы не нужны. Плата — линейная сходимость: на каждом шаге ошибка умножается примерно на одно и то же число. Оно тем меньше, чем меньше $\lambda^\ast$.

Первый шаг метода из $\theta_c$ — самое короткое решение уравнения $Jp=t-e(\theta_c)$, то есть $p=J^{+}\big(t-e(\theta_c)\big)$ с псевдообратной матрицей $J^{+}$ (`np.linalg.pinv`). В робототехнике это классический шаг обратной кинематики через псевдообратную матрицу якобиана.

**Задание 2 (1 балл): `gn_kkt(kkt, jac_r)`.** Функция берёт `kkt` задачи и функцию `jac_r(x)`, которая возвращает $J_r$. Вернуть она должна новую функцию `kkt_gn(x, lam)`: то же, что `kkt`, но вместо $H$ — матрица $J_r^\top J_r$. Тогда `newton_lagrange(gn_kkt(kkt, jac_r), x0, lam0)` — это обобщённый Гаусс–Ньютон. Множитель $\lambda$ в шаге сокращается, поэтому система та же, что в §5.

In [ ]:
def gn_kkt(kkt, jac_r):
    """kkt для обобщённого Гаусса–Ньютона: вместо гессиана лагранжиана H — матрица J_r^T J_r."""
    # TODO: ваш код
    raise NotImplementedError

In [ ]:
# проверка задания 2: окружность лекции 9 §5, невязка r = x - a, J_r = I, старт (0.6, 0.8)
eye2 = lambda x: np.eye(2)
h = newton_lagrange(gn_kkt(circle_kkt(np.array([2.0, 1.0])), eye2), [0.6, 0.8], [0.0], maxit=1)
assert np.allclose(h[1][0], [1.4, 0.2]), f"при a = (2, 1) первый шаг — проекция a на касательную, точка (1.4, 0.2); у вас {h[1][0]}"
a = np.array([0.96, 0.72])                                          # |a| = 1.2, lambda* = -0.2
h = newton_lagrange(gn_kkt(circle_kkt(a), eye2), [0.6, 0.8], [0.0])
ang = [np.arctan2(x[1], x[0]) - np.arctan2(a[1], a[0]) for x, _, _ in h]     # ошибка угла
assert abs(ang[5] / ang[4] + 0.2) < 1e-3, f"в конспекте ошибка угла умножается на lambda* = -0.2, у вас на {ang[5] / ang[4]:.3f}"
print(f"задание 2: ok;  первый шаг в (1.4, 0.2), отношение ошибок угла {ang[5] / ang[4]:.4f}")

**(а) Картинка 3 (2 балла): чем дальше цель, тем медленнее.** Функция `target_rotated(beta)` поворачивает нынешний конец руки $e(\theta_c)$ вокруг основания на угол $\beta$. Для $\beta=0.2,\ 0.6,\ 1.0$ рад запустите из $\theta_c$ оба метода: Ньютона–Лагранжа и Гаусса–Ньютона с `jac_r = lambda th: np.eye(3)`. Ответ $\theta^\ast$ возьмите у Ньютона–Лагранжа. Посчитайте ошибки $\lVert\theta_k-\theta^\ast\rVert$ обоих методов и нарисуйте `plot_errors(err_gn, err_nl, labels)`.

Ответьте: сколько итераций нужно каждому методу? Как выглядят кривые Гаусса–Ньютона в логарифмической шкале и что это значит? Почему для дальней цели метод медленнее?

*Полный балл:* картинка; итерации обоих методов; объяснение «прямая в логарифмической шкале — ошибка умножается на постоянное число, сходимость линейная» против квадратичной у Ньютона–Лагранжа; связь скорости с $\lVert\lambda^\ast\rVert$. *Половина:* картинка и числа без объяснения.

In [ ]:
# TODO: ваш код
# для beta из (0.2, 0.6, 1.0): t = target_rotated(beta)
#   hn = newton_lagrange(arm_kkt(t), TH_C, np.zeros(2));  hg = newton_lagrange(gn_kkt(arm_kkt(t), lambda th: np.eye(3)), TH_C, np.zeros(2))
#   theta* = hn[-1][0]; ошибки |theta_k - theta*| для обоих методов — в списки err_gn и err_nl; сохраните (t, theta*, lambda*) для пункта (б)
# картинка: plot_errors(err_gn, err_nl, [f"поворот {b} рад" for b in betas]); plt.show()


_Ваш ответ:_ TODO

**(б) Множитель ошибки (2 балла).** Вблизи ответа Гаусс–Ньютон умножает ошибку вдоль касательного вектора $Z$ на число

$$
q=1-Z^\top HZ=\lambda_1\,Z^\top\nabla^2e_1\,Z+\lambda_2\,Z^\top\nabla^2e_2\,Z .
$$

Здесь $H$ и $Z$ берутся в ответе $(\theta^\ast,\lambda^\ast)$, а $\lVert Z\rVert=1$. Посчитайте $q$ для трёх целей пункта (а). Измерьте отношение $e_{k+1}/e_k$ ошибок Гаусса–Ньютона на последних итерациях, где ошибка ещё больше $10^{-9}$. Сведите в таблицу: $\beta$, $\lVert\lambda^\ast\rVert$, $q$, измеренное отношение.

На бумаге примените ту же формулу к окружности лекции 9 §5: $f=\tfrac12\lVert x-a\rVert^2$, $g=\tfrac12\big(\lVert x\rVert^2-1\big)$. Покажите, что там $q=\lambda^\ast$, как в конспекте. Ответьте: почему $q$ пропорционально множителям и что означает знак минус у $q$?

*Полный балл:* таблица, где $\lvert q\rvert$ совпадает с измеренным отношением; вывод $q=\lambda^\ast$ на окружности; объяснение: $q$ — выброшенная кривизна ограничений, она входит в $H$ с множителями; смысл знака. *Половина:* таблица без вывода.

In [ ]:
# TODO: ваш код
# для каждой цели из (а): gf, J, H, g = arm_kkt(t)(theta*, lambda*); Z = null_space(J); q = 1 - (Z.T @ H @ Z).item()
# измеренное отношение: e[k] / e[k - 1] для последней итерации k, где ошибка Гаусса–Ньютона e[k] > 1e-9


_Ваш ответ:_ TODO

## Бонус (+2 балла). Далёкая цель по кадрам

**Главная идея.** Метод Ньютона–Лагранжа сходится быстро только из близкого старта. Поэтому далёкую цель подают по кадрам. Каждый кадр сдвигает цель немного, а старт — поза прошлого кадра. Так работают контроллеры роботов в реальном времени.

Готовая ячейка ниже ставит цель сразу на угол $\beta=1.5$ рад и запускает из $\theta_c$ наш метод и IPOPT. Оба приходят в решения системы, но суставы в них повёрнуты гораздо сильнее, чем нужно.

Напишите `track(targets, theta0, use_gn=True)`. Для каждой цели $t_k$ из списка `targets`:

1. задача кадра — та же, но текущая поза в $f$ — поза прошлого кадра: `arm_kkt(t_k, pose)`;
2. при `use_gn=True` — обобщённый Гаусс–Ньютон через ваш `gn_kkt`, иначе — Ньютон–Лагранж;
3. старт — поза прошлого кадра, множители — нули.

Верните массив поз (первая — `theta0`) и список чисел итераций по кадрам.

**(а) Картинка 4 (1 балл).** Ячейка после вашей функции проведёт руку к той же цели за $15$ кадров по $0.1$ рад. Она запустит оба метода и нарисует `plot_track`. Ответьте: сколько итераций на кадр нужно каждому методу и чему равно $f$ итоговой позы, считая от $\theta_c$?

**(б) (1 балл).** Почему прямой ход увёл метод в такую позу? Почему по кадрам хватает нескольких итераций? Почему Гауссу–Ньютону здесь нужно ненамного больше итераций, чем Ньютону–Лагранжу?

*Балл:* (а) — проверка проходит, картинка и числа; (б) — объяснение через локальную сходимость, малые $\lambda^\ast$ кадра и малое $q$. Бонус необязателен: если пропускаете, ячейки бонуса оставьте как есть.

In [ ]:
# бонус: цель сразу на 1.5 рад — наш метод и IPOPT из текущей позы
T_FAR = target_rotated(1.5)
h_far = newton_lagrange(arm_kkt(T_FAR), TH_C, np.zeros(2))
th_ip_far, _, it_ip_far = solve_ipopt(T_FAR)
for name, th, k in (("Ньютон–Лагранж", h_far[-1][0], len(h_far) - 1), ("IPOPT", th_ip_far, it_ip_far)):
    print(f"{name:15s}: итераций {k:2d}, theta = {th.round(2)} рад, это {(th / (2 * np.pi)).round(2)} оборота; "
          f"f = {0.5 * np.sum((th - TH_C) ** 2):.1f}")

In [ ]:
# бонус (необязательно): напишите здесь track(targets, theta0, use_gn=True)


In [ ]:
if "track" in globals():                # бонус (а): выполняется, только если вы написали track
    targets = [target_rotated(0.1 * k) for k in range(1, 16)]
    poses_gn, its_gn = track(targets, TH_C)
    poses_nl, its_nl = track(targets, TH_C, use_gn=False)
    assert np.allclose(arm_end(poses_gn[-1]), T_FAR, atol=1e-8) and np.allclose(poses_gn[-1], poses_nl[-1], atol=1e-6), \
        "в последнем кадре конец руки должен быть в цели T_FAR, а оба метода — в одной позе"
    print("итераций на кадр: Гаусс–Ньютон", its_gn, ", Ньютон–Лагранж", its_nl)
    print(f"итоговая поза {poses_gn[-1].round(3)}, f от TH_C = {0.5 * np.sum((poses_gn[-1] - TH_C) ** 2):.2f}")
    plot_track(poses_gn, targets, its_gn, its_nl); plt.show()
else:
    print("бонус пропущен")

_Бонус (необязательно):_ ваш ответ

_Бонус (необязательно):_ ваш ответ